# Beamforming

## What is beamforming

Beamforming combines the signals from several antennas so that the group acts like one large antenna that can be pointed electronically.
- **Steering without moving parts:** each element's signal is delayed or phase-shifted before summing, so the beam can be pointed (and re-pointed in microseconds) without physically moving anything.
- **Gain and directivity:** signals from the steered direction add constructively, giving more gain and a narrower beam than any single element.
- **Interference rejection:** signals from other directions add out of phase and are suppressed, and nulls can be placed on purpose toward jammers or interferers.
- **Works both ways:** the same idea applies when receiving (combining incoming signals) and transmitting (shaping where the energy goes).

Examples:
- **Phased array radars:** military and weather radars that scan the sky electronically
- **Starlink "Dishy":** a flat phased array that tracks fast-moving satellites without a motor doing the pointing
- **5G cell towers:** massive MIMO arrays that aim separate beams at individual users
- **Radio astronomy:** arrays like the Very Large Array (VLA) in New Mexico combine widely spaced dishes into one high-resolution instrument
- **Microphone arrays:** smart speakers, conference phones, and headset mics use beamforming to focus on the speaker's voice and reject background noise

## How does it work 
A plane wave arriving from off boresight reaches each antenna at a slightly different time. Before summing the antennas together, we delay each one (or, for a narrowband signal, multiply it by a phase offset) so that signals from the direction we care about line up and add constructively. Signals from other directions don't line up, so they partially cancel. Choosing the delays/phases is what "points" the beam in a specific direction.

This is very similar to a finite impulse response filter: a set of weighted taps summed together. But instead of the taps being samples spaced in time, they are antennas spaced in distance, so the array is a **spatial filter** that passes some directions and rejects others. Remember that signals travel at the speed of light, so distance traveled and time offset are equivalent.

Lets talk about the math given a uniform linear array of n antennas

$s$ spacing between antennas in units of wavelength $\lambda$

The position of each element $n$ is 

$$
d_n = s  n  \lambda
$$

For a signal coming in at an angle $\theta$ degrees off of boresight the extra distance needed to travel to each element is:

$$
\Delta P_n = d_n \sin( \theta )
$$

Then the phase offset of the incoming signal at each antenna is 

$$
\Delta\phi = 2 \pi \frac{\Delta P_n}{\lambda}
$$

This means that even though the signals are delayed in time we can make use of the fact that they share a common carrier signal and reference them in terms of difference of phase of arrival 

### Steering vector

To point the beam at an angle $\theta_0$ we want to undo the phase offset that a signal from $\theta_0$ picks up at each element. So each element gets multiplied by a complex weight with the opposite phase:

$$
w_n = e^{-j 2 \pi s n \sin(\theta_0)}
$$

Stacking these for all $N$ elements gives the steering vector

$$
\mathbf{w} = \begin{bmatrix} 1 & e^{-j 2 \pi s \sin(\theta_0)} & e^{-j 2 \pi s 2 \sin(\theta_0)} & \cdots & e^{-j 2 \pi s (N-1) \sin(\theta_0)} \end{bmatrix}^T
$$

A signal arriving from some other angle $\theta$ shows up across the array with phases

$$
a_n(\theta) = e^{j 2 \pi s n \sin(\theta)}
$$

Weighting each element and summing gives the **array factor**, which is how strongly the array responds to a signal from direction $\theta$:

$$
AF(\theta) = \sum_{n=0}^{N-1} w_n \, a_n(\theta) = \sum_{n=0}^{N-1} e^{j 2 \pi s n \left(\sin(\theta) - \sin(\theta_0)\right)}
$$

When $\theta = \theta_0$ every term is $1$, so they all add up in phase and $|AF| = N$, the peak of the beam. The code below computes exactly this: `steeringPhase` is the phase of $w_n$, `observedPhase` is the phase of $a_n(\theta)$, and `factor` is $AF(\theta)$, which then gets normalized and plotted in dB.



## Things to keep in mind
- Just like in time based signals nyquist sampling criteria applies. If your antennas are spaced by more than $\lambda/2$ then you will encounter aliasing 

## Demo below

The demo below allows you to set any number of elements, any spacing and any pointing angle. Here are a few things to try

- Increase the number of elements. Notice how more elements equals a tighter beam. This is analogous to more taps in a FIR resulting in the ability to have a tighter frequency cutoff.
- Change the spacing. Start with $\lambda/2$ but notice how when you raise the spacing above $\lambda/2$ that a **second** beam appears in the beam pattern. One beam will be at your desired steering angle, the other is the aliased beam which is often called a "grating lobe"
- Set a high number of elements and a spacing of $\lambda/2$ and start playing with the steering angle. Notice that the beam becomes less tight the further you steer off boresight.

In [25]:
# Packages not bundled with Pyodide are installed from PyPI in the browser
%pip install -q plotly nbformat ipywidgets


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [26]:
# imports here
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets


In [27]:
style = {"description_width": "initial"}
layout = widgets.Layout(width="500px")
# number of array elements
n_slider = widgets.IntSlider(value=2, min=2, max=30, step=1, description="Elements (n)",
                             continuous_update=False, style=style, layout=layout)
# element spacing in wavelengths
spacing_slider = widgets.FloatSlider(value=0.5, min=0.1, max=1.0, step=0.05, description="Spacing (λ)",
                                     continuous_update=False, style=style, layout=layout)
# pointing angle from boresight of the array
angle_slider = widgets.FloatSlider(value=0, min=-90, max=90, step=1, description="Steer angle (°)",
                                   continuous_update=False, style=style, layout=layout)

In [28]:
def array_pattern(n, spacing, angleDeg):
    angleRad = np.pi * angleDeg / 180.0

    elementId = np.arange(n)
    elementPos = elementId * spacing

    steeringPath = elementPos * np.sin(angleRad)
    steeringPhase = -2 * np.pi * steeringPath

    # Turn it into a column
    steeringPhaseCol = steeringPhase[:, None]

    observationDeg = np.linspace(-90, 90, 1000)
    observationRad = np.pi * observationDeg / 180.0

    observedPhase = (
        2 * np.pi
        * elementPos[:, None]
        * np.sin(observationRad)[None, :]
    )

    totalPhase = steeringPhaseCol + observedPhase

    combinedComplex = np.exp(1j * totalPhase)
    factor = np.sum(combinedComplex, axis=0)

    pattern = np.abs(factor)
    patternNormalized = pattern / np.max(pattern)
    patternDB = 20 * np.log10(
        np.maximum(patternNormalized, 1e-12)
    )

    return observationDeg, patternDB

In [29]:
def geometry_fig(n, spacing, angleDeg):
    """Plane wave arriving at angle theta on the array, showing the extra path ΔP_n to each element."""
    th = np.radians(angleDeg)
    d = np.arange(n) * spacing                # element positions d_n (λ)
    u = np.array([-np.sin(th), np.cos(th)])   # unit vector pointing back toward the source
    v = np.array([np.cos(th), np.sin(th)])    # direction along the wavefront
    dP = d * np.sin(th)                       # extra path ΔP_n = d_n sin(θ) (λ)
    span = max(d[-1], 1.0)
    L = span + 1                              # how far past the wavefront to draw the rays
    fig = go.Figure()

    # array baseline
    fig.add_trace(go.Scatter(x=[-0.1 * span, d[-1] + 0.1 * span], y=[0, 0], mode="lines",
                             line=dict(color="lightgray", width=1), hoverinfo="skip", showlegend=False))

    # boresight and θ arc at element 0
    fig.add_trace(go.Scatter(x=[0, 0], y=[0, 0.8 * L], mode="lines+text", text=["", "boresight"],
                             textposition="top center", line=dict(color="gray", dash="dot"),
                             hoverinfo="skip", showlegend=False))
    r, phi = 0.3 * L, np.linspace(0, th, 30)
    fig.add_trace(go.Scatter(x=-r * np.sin(phi), y=r * np.cos(phi), mode="lines",
                             line=dict(color="gray"), hoverinfo="skip", showlegend=False))
    fig.add_annotation(x=-1.2 * r * np.sin(th / 2), y=1.2 * r * np.cos(th / 2),
                       text=f"θ = {angleDeg:.0f}°", showarrow=False)

    # incoming rays, each ending at its element
    for dn, dPn in zip(d, dP):
        start = np.array([dn, 0]) + (dPn + L) * u
        fig.add_annotation(x=dn, y=0, ax=start[0], ay=start[1], xref="x", yref="y",
                           axref="x", ayref="y", showarrow=True, arrowhead=2,
                           arrowwidth=1.5, arrowcolor="#1f77b4")
    fig.add_trace(go.Scatter(x=[None], y=[None], mode="lines", name="Incoming rays",
                             line=dict(color="#1f77b4")))

    # wavefront through element 0
    w = np.array([-0.15 * span, d[-1] * np.cos(th) + 0.15 * span])
    fig.add_trace(go.Scatter(x=w * v[0], y=w * v[1], mode="lines", name="Wavefront at element 0",
                             line=dict(color="#2ca02c", dash="dash"), hoverinfo="skip"))

    # extra path ΔP_n from the wavefront down to each element
    xs, ys = [], []
    for dn, dPn in zip(d, dP):
        xs += [dn, dn + dPn * u[0], None]
        ys += [0, dPn * u[1], None]
    fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", name="ΔP<sub>n</sub> = d<sub>n</sub> sin θ",
                             line=dict(color="#d62728", width=4), hoverinfo="skip"))

    # right angle where the last element's ray meets the wavefront
    if abs(dP[-1]) > 1e-9:
        q, a = np.array([d[-1], 0]) + dP[-1] * u, 0.04 * span
        toward = -np.sign(dP[-1]) * u
        sq = np.array([q - a * v, q - a * v + a * toward, q + a * toward])
        fig.add_trace(go.Scatter(x=sq[:, 0], y=sq[:, 1], mode="lines", line=dict(color="black", width=1),
                                 hoverinfo="skip", showlegend=False))
        mid = np.array([d[-1], 0]) + 0.5 * dP[-1] * u
        fig.add_annotation(x=mid[0], y=mid[1], text=f"ΔP<sub>{n-1}</sub> = {dP[-1]:.2f}λ",
                           showarrow=False, xshift=50, font=dict(color="#d62728"))

    # elements
    fig.add_trace(go.Scatter(x=d, y=np.zeros(n), mode="markers", name="Elements",
                             marker=dict(symbol="triangle-up", size=12, color="black"),
                             customdata=np.stack([np.arange(n), d, dP], axis=1),
                             hovertemplate="n=%{customdata[0]}<br>d<sub>n</sub>=%{customdata[1]:.2f}λ"
                                           "<br>ΔP<sub>n</sub>=%{customdata[2]:.2f}λ<extra></extra>"))
    fig.add_annotation(x=d[-1] / 2, y=0, yshift=-25, showarrow=False,
                       text=f"d<sub>{n-1}</sub> = {d[-1]:.2f}λ  (spacing {spacing:.2f}λ)")

    fig.update_layout(title="Array geometry", width=700, height=500, template="plotly_white",
                      legend=dict(x=1.02, y=1))
    fig.update_xaxes(title="Position along array (λ)", zeroline=False)
    fig.update_yaxes(title="(λ)", zeroline=False, scaleanchor="x", scaleratio=1)
    return fig

In [30]:
out_geo = widgets.Output()
out = widgets.Output()


def update(_=None):
    observationDeg, patternDB = array_pattern(n_slider.value, spacing_slider.value, angle_slider.value)
    fig = go.Figure(go.Scatter(x=observationDeg, y=patternDB, mode="lines", name="Array factor"))
    fig.update_layout(title="Array pattern", width=700, height=450, template="plotly_white")
    fig.update_xaxes(title="Observation angle (°)", range=[-90, 90])
    fig.update_yaxes(title="Normalized pattern (dB)", range=[-50, 1])
    with out_geo:
        out_geo.clear_output(wait=True)
        geometry_fig(n_slider.value, spacing_slider.value, angle_slider.value).show()
    with out:
        out.clear_output(wait=True)
        fig.show()


# drop callbacks from earlier runs of this cell so each slider move draws once
for s in (n_slider, spacing_slider, angle_slider):
    s.unobserve_all(name="value")
    s.observe(update, names="value")

display(widgets.VBox([n_slider, spacing_slider, angle_slider, out_geo, out]))

update()